# Build the copy of the R library for Google Colab (instructor only)

This notebook installs, **once** in Google Colab, all the R packages used in the exposome tutorial and saves a copy of the R library as a single file (`r_library_colab.tar.gz`). During the session, the tutorial downloads this file instead of installing everything from scratch (Option A in Section 1 of the tutorial).

**When:** shortly before the session (e.g., the day before). The copy only works with the R version that Colab uses at that moment. If Colab updates R, the copy has to be rebuilt.

**Steps**
1. Open this notebook in Google Colab with an **R** runtime (*Runtime → Change runtime type → R*).
2. Run all the cells (≈ 30–40 minutes, mainly the Bioconductor packages, `rexposome`, `tidyverse` and `caret`).
3. Download `r_library_colab.tar.gz` from the **Files** panel on the left (folder `/content`).
4. In GitHub, go to the tutorial repository → **Releases** → **Draft a new release**:
   * Tag: `colab-lib` (it must be exactly this, because the tutorial downloads from `.../releases/download/colab-lib/r_library_colab.tar.gz`).
   * Attach `r_library_colab.tar.gz` and publish the release. (Release files can be up to 2 GB, unlike files in the repository.)
5. To update it later: edit the same release, delete the old file and upload the new one.
6. Test it: open the tutorial in a new Colab session and run Option A and the loading cell.

In [ ]:
# Install all the packages into a separate folder (r_lib)
t0 <- Sys.time()

lib_dir <- "/content/r_lib"
dir.create(lib_dir, showWarnings = FALSE)
.libPaths(c(lib_dir, .libPaths()))  # new packages are installed in r_lib

if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager", lib = lib_dir)
options(repos = BiocManager::repositories())  # CRAN + Bioconductor

# Same packages as in the tutorial
# Section 1: packages installed with the BiocManager loop
setup_pkgs <- c("Biobase", "mice", "MultiDataSet", "lsr", "FactoMineR",
                "stringr", "circlize", "corrplot", "ggplot2", "reshape2", "pryr",
                "scales", "imputeLCMD", "scatterplot3d", "glmnet", "gridExtra",
                "Hmisc", "gplots", "gtools", "S4Vectors", "qgcomp")
# Section 1 (pacman::p_load) and Section 4 (variable selection, clustering, mixtures)
analysis_pkgs <- c("pacman", "tidyverse", "RColorBrewer", "skimr", "bkmr", "gWQS",
                   "ggridges", "caret", "partDSA", "factoextra", "dotwhisker")
# MASS, grid and cluster come with R, so they are not installed here

BiocManager::install(c(setup_pkgs, analysis_pkgs), lib = lib_dir, ask = FALSE, update = FALSE)

# rexposome from GitHub, as in the tutorial
if (!requireNamespace("remotes", quietly = TRUE)) install.packages("remotes", lib = lib_dir)
remotes::install_github("isglobal-brge/rexposome", lib = lib_dir, upgrade = "never")

cat(sprintf("Installation time: %.1f minutes\n", as.numeric(difftime(Sys.time(), t0, units = "mins"))))

In [ ]:
# Check that every package can be loaded
all_pkgs <- c(setup_pkgs, analysis_pkgs, "rexposome")

loaded_ok <- sapply(all_pkgs, function(pkg) {
  suppressPackageStartupMessages(requireNamespace(pkg, quietly = TRUE))
})
print(loaded_ok)
if (!all(loaded_ok)) cat("FIX BEFORE CONTINUING. Not loaded:", names(loaded_ok)[!loaded_ok], "\n")

In [ ]:
# Save the R version used and compress the library into one file
writeLines(R.version.string, file.path(lib_dir, "R_VERSION.txt"))

lib_tar <- "/content/r_library_colab.tar.gz"
system2("tar", c("-czf", lib_tar, "-C", "/content", "r_lib"))

cat("R version:", R.version.string, "\n")
cat("Packages in the copy:", length(list.dirs(lib_dir, recursive = FALSE)), "\n")
cat("File:", lib_tar, "-", round(file.size(lib_tar) / 1024^2), "MB\n")